# 📕 Notebook 3 — gemma-4-E4B-it
## Extraction de définitions juridiques maritimes + Benchmark

### Pourquoi Gemma-4-9B ?
- **Référence Meta officielle** : Gemma-4 est le modèle de référence du projet — permet la comparaison directe avec votre extraction LLaMA existante
- **Fort suivi d'instructions** : Gemma-4-Instruct est optimisé par RLHF pour suivre des instructions complexes, ce qui est critique pour l'extraction de définitions avec exclusions et références
- **Tokenizer amélioré** : vocabulaire 128K tokens (vs 32K pour LLaMA-2), meilleure gestion du français et des caractères spéciaux juridiques
- **Fenêtre 128K** : identique à Qwen3, supérieure à Ministral-8B
- **Benchmark interne** : ce notebook intègre aussi la comparaison finale des 3 modèles


In [2]:
!pip install -q --upgrade transformers accelerate bitsandbytes pdfplumber PyMuPDF pandas tqdm matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [3]:
import torch
import json
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import pdfplumber
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU : Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ─── Si nécessaire, authentification HuggingFace ─────────────────────────────
from huggingface_hub import login
import os

# Remplacez 'hf_VOTRE_TOKEN_ICI' par le token que vous avez copié depuis Hugging Face.
# Si vous l'avez stocké dans les secrets Colab, vous pouvez utiliser os.environ.get("HF_TOKEN").
#login(token="hf_VOTRE_TOKEN_ICI")

# Alternativement, si vous utilisez les secrets Colab (recommandé pour la sécurité) :
from google.colab import userdata
login(token=userdata.get('HF_TOKEN')) # Assurez-vous d'avoir un secret nommé HF_TOKEN avec votre token dedans

In [13]:
import torch
import json
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import pdfplumber
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# --- Configuration ───────────────────────────────────────────────────────────
MODEL_NAME = "google/gemma-4-E4B-it"
# Note : nécessite un token HuggingFace avec accès Gemma-4
# from huggingface_hub import login; login(token="hf_VOTRE_TOKEN")

OUTPUT_DIR = Path("/content/drive/MyDrive/data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_DIR = Path("/content/drive/MyDrive/data/output/comparison")
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS = [
    {"id": "I001", "label": "Chalutage de fond", "lang": "fr",
     "path": "/content/clalut_fond.pdf"},
    {"id": "I002", "label": "Chasse à la baleine", "lang": "en",
     "path": "/content/drive/MyDrive/data/raw/Baleine/ICRW_convention.pdf"},
    {"id": "I003", "label": "Construction littorale", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/construction_sur_le_littorale/protocole_GIZC.pdf"},
    {"id": "I004", "label": "Extraction de sable", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/extraction_Sable/convention_cbd.pdf"},
    {"id": "I005a", "label": "Rejet hydrocarbures - Protocole", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/Rejet_dhydrocarbure/94ig4_4_protocol_fre.pdf"},
    {"id": "I005b", "label": "Rejet hydrocarbures - MARPOL Annexe I", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/Rejet_dhydrocarbure/Annexe1consolidee_Marpol.pdf"},
    {"id": "I006", "label": "Sacs plastique - MARPOL Annexe V", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/Sac_plastique/ANNEXEV_marpol.pdf"},
    {"id": "I007", "label": "Peintures TBT - AFS Convention", "lang": "en",
     "path": "/content/drive/MyDrive/data/raw/TBT/AFS_convention.pdf"},
    {"id": "I008", "label": "Oiseaux marins - CMS", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/Oiseaux_marin/CMS_oiseau_marin.pdf"},
]

GPU : Tesla T4


In [7]:
# ─── Extraction du texte PDF ─────────────────────────────────────────────────
def extract_pdf_text(pdf_path: str, max_chars: int = 12000) -> str:
    """
    Extrait le texte d'un PDF avec pdfplumber (meilleur pour les tableaux/colonnes).
    Fallback sur PyMuPDF si le PDF est scanné ou corrompu.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if len(text.strip()) < 100:  # PDF scanné → fallback PyMuPDF
            raise ValueError("Texte trop court, possible scan")
    except Exception as e:
        print(f"  [WARNING] pdfplumber échoué ({e}), fallback PyMuPDF...")
        try:
            doc = fitz.open(pdf_path)
            for page in doc:
                text += page.get_text() + "\n"
        except Exception as e2:
            print(f"  [ERROR] PyMuPDF aussi échoué : {e2}")
            return ""

    # Nettoyage basique
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text[:max_chars]  # Limite réduite pour éviter CUDA OOM


# Test sur le premier document
test_doc = DOCUMENTS[0]
if Path(test_doc["path"]).exists():
    sample = extract_pdf_text(test_doc["path"])
    print(f"Extrait ({len(sample)} chars) :\n{sample[:500]}")
else:
    print(f"[INFO] Fichier non trouvé : {test_doc['path']} (normal en dev local)")

[INFO] Fichier non trouvé : /content/clalut_fond.pdf (normal en dev local)


In [8]:
def extract_relevant_sections(text: str) -> str:
    collected = []

    # ── Stratégie 1 : sections titrées (toutes casses) ───────────────────────
    # Couvre : DEFINITIONS, Definitions, definitions, INTERPRETATION, etc.
    title_patterns = [
        r'((?:Article|ARTICLE|Rule|RULE|Règle 1|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH|Para\.?)\s*[\dIVXivx]+\s*[-–.]?\s*(?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\b.*?)(?=\n(?:Article|ARTICLE|Rule|RULE|Règle|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH)\s*[\dIVXivx]+\b)',
        r'((?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\s*\n+.*?)(?=\n{2,}[A-ZÀÂÉÈÊËÎÏÔÙÛÜ]{2}|\nArticle|\nARTICLE|\nRule|\nRULE|\nSection|\nSECTION)',
    ]
    for pattern in title_patterns:
        matches = re.findall(pattern, text, re.DOTALL)
        collected.extend(matches)

    # ── Stratégie 2 : pattern "terme" means/signifie (toutes casses) ─────────
    trigger = (
        r'(?:'
        r'[Mm][Ee][Aa][Nn][Ss]?\b'                          # means / mean / MEANS
        r'|[Ss][Hh][Aa][Ll][Ll]\s+[Mm][Ee][Aa][Nn]\b'      # shall mean / SHALL MEAN
        r'|[Rr][Ee][Ff][Ee][Rr][Ss]?\s+[Tt][Oo]\b'         # refers to / REFERS TO
        r'|[Ii][Ss]\s+[Dd][Ee][Ff][Ii][Nn][Ee][Dd]\s+[Aa][Ss]\b'  # is defined as
        r'|[Ss][Ii][Gg][Nn][Ii][Ff][Ii][Ee]\b'             # signifie / SIGNIFIE
        r'|[Dd][Éé][Ss][Ii][Gg][Nn][Ee]\b'                 # désigne / DÉSIGNE
        r'|[Ss][\""][Ee][Nn][Tt][Ee][Nn][Dd]\s+(?:[Dd][Ee]|[Cc][Oo][Mm][Mm][Ee])\b'  #entend de/comme
        r'|[Ee][Ss][Tt]\s+[Dd][Éé][Ff][Ii][Nn][Ii]\s+[Cc][Oo][Mm][Mm][Ee]\b'        # est défini comme
        r'|[Aa][Uu][Xx]\s+[Ff][Ii][Nn][Ss]\s+(?:[Dd][Ee]\s+[Ll][Aa]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt][Ee]|[Dd][Uu]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt])\b'  # aux fins de la présente
        r')'
    )

    # Terme entre guillemets (droits, français, allemands) ou TOUT EN MAJUSCULES
    term = (
        r'(?:'
        r'«\s*[^»]{1,80}\s*»'       # «terme»
        r'|"\s*[^"]{1,80}\s*"'      # "terme"
        r'|"\s*[^"]{1,80}\s*"'      # "terme" (guillemets typographiques)
        r'|„[^"]{1,80}"'            # „terme" (allemand)
        r"|'[^']{1,80}'"            # 'terme'
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][A-ZÀÂÉÈÊËÎÏÔÙÛÜ\s\-]{2,50}(?=\s)'  # TOUT EN MAJUSCULES
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][a-zàâéèêëîïôùûüa-z\s\-]{2,50}(?=\s)'  # Première lettre majuscule
        r')'
    )

    pattern_inline = re.compile(
        rf'({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*\n|\Z|{term}\s*(?:\([^)]*\))?\s*{trigger})',
        re.DOTALL
    )
    matches2 = pattern_inline.findall(text)
    collected.extend(matches2)

    # ── Stratégie 3 : listes numérotées/lettrées ─────────────────────────────
    pattern_list = re.compile(
        rf'(?:^\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s+)({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s|\n\s*\n|\Z)',
        re.DOTALL | re.MULTILINE
    )
    matches3 = pattern_list.findall(text)
    collected.extend(matches3)

    # ── Résultat ──────────────────────────────────────────────────────────────
    if collected:
        seen = set()
        deduped = []
        for item in collected:
            key = item.strip()[:100]
            if key not in seen:
                seen.add(key)
                deduped.append(item.strip())

        result = "\n\n".join(deduped)

        if len(result) < 200:
            print(f"  [WARNING] Sections trop courtes ({len(result)} chars), chunking document entier")
            return text

        print(f"  [INFO] {len(deduped)} bloc(s) trouvé(s) → {len(result)} chars ciblés")
        return result

    print(f"  [WARNING] Aucune section détectée, chunking document entier")
    return text

In [9]:
# ─── Prompts Gemma-4 (format <|begin_of_text|> géré par apply_chat_template)
SYSTEM_FR = (
    "Tu es un juriste spécialisé en droit maritime international. "
    "Ta tâche est d'extraire les définitions juridiques exactes d'un document. "
    "Tu réponds UNIQUEMENT en JSON valide, sans aucun texte supplémentaire."
)

SYSTEM_EN = (
    "You are a legal expert specializing in international maritime law. "
    "Your task is to extract exact legal definitions from a document. "
    "You respond ONLY in valid JSON, without any additional text."
)

USER_FR = """Extraire toutes les définitions du document présentes dans les sections
« Définitions », « Interprétation » ou introduites par « signifie », « désigne »,
« s'entend de », « est défini comme », « Aux fins de la présente »,
ou toute liste numérotée/lettrée sous un article définitionnel.

Pour chaque terme, restituer le texte EXACT sans paraphrase :
- le terme exact
- sa définition littérale complète
- le nom scientifique en parenthèses si présent
- la référence article/règle/paragraphe
- toute exclusion/restriction mentionnée

Format de sortie OBLIGATOIRE — tableau JSON uniquement :
[
  {{"terme": "...", "definition": "...", "nom_scientifique": null, "reference": "...", "exclusions": null, "langue": "fr"}}
]

DOCUMENT :
{text}"""

USER_EN = """Extract all definitions from this document present in sections
"Definitions", "Interpretation", or introduced by "means", "shall mean",
"refers to", "is defined as", "for the purposes of",
or any numbered/lettered list under a definitional article.

For each term, provide the EXACT text without paraphrase:
- the exact term
- its complete literal definition
- scientific name in parentheses if present
- article/rule/paragraph reference
- any exclusion/restriction mentioned

MANDATORY output format — JSON array only:
[
  {{"terme": "...", "definition": "...", "nom_scientifique": null, "reference": "...", "exclusions": null, "langue": "en"}}
]

DOCUMENT:
{text}"""

def get_messages_llama(lang: str, text: str) -> list:
    if lang == "fr":
        return [
            {"role": "system", "content": SYSTEM_FR},
            {"role": "user", "content": USER_FR.format(text=text)}
        ]
    else:
        return [
            {"role": "system", "content": SYSTEM_EN},
            {"role": "user", "content": USER_EN.format(text=text)}
        ]

In [1]:
!pip install --upgrade git+https://github.com/huggingface/transformers.git accelerate bitsandbytes

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-fuvon_e2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-fuvon_e2
  Resolved https://github.com/huggingface/transformers.git to commit a553395766001116a719c82870171f8d6b458c98
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11381798 sha256=ff9222bcc24b387a6f73ffd56589f2e1ff597b3464d9d7be1abca9cfedb27105
  Stored in directory: /tmp/pip-ephem-wheel-cache-dopgkjao/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.3
    Uninstalling transformers-5.5.3:
      Successfully uninstalled transformers-5.5.3


In [10]:
# ─── Chargement Gemma-4-9B en 4-bit ───────────────────────────────────────
print(f"Chargement de {MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("✅ Gemma-4-9B chargé")
if torch.cuda.is_available():
    print(f"VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Chargement de google/gemma-4-E4B-it...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

✅ Gemma-4-9B chargé
VRAM : 9.30 GB


In [11]:
# ─── Inférence Gemma-4 ─────────────────────────────────────────────────────
def extract_definitions_llama(doc_text: str, lang: str, doc_id: str) -> list:
    messages = get_messages_llama(lang, doc_text)

    # Correction : on extrait les ['input_ids'] du résultat du tokenizer
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    input_ids = inputs if torch.is_tensor(inputs) else inputs["input_ids"]
    input_len = input_ids.shape[1]
    attention_mask = torch.ones_like(input_ids)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<eos>")
    ]
    terminators = [t for t in terminators if t is not None]

    start_time = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=3500,
            eos_token_id=terminators,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start_time

    new_tokens = outputs[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"  Génération : {elapsed:.1f}s, {len(new_tokens)} tokens")

    return parse_json_response(response, doc_id)

def parse_json_response(response: str, doc_id: str) -> list:
    clean = re.sub(r'```(?:json)?\s*', '', response)
    clean = re.sub(r'```\s*', '', clean).strip()

    for strategy in [lambda s: json.loads(s),
                     lambda s: json.loads(re.search(r'(\[.*\])', s, re.DOTALL).group(1))]:
        try:
            data = strategy(clean)
            if isinstance(data, list):
                for item in data:
                    item['doc_id'] = doc_id
                    item['modele'] = 'gemma-4-9b-it'
                return data
        except Exception:
            continue

    objects = re.findall(r'\{[^{}]+\}', clean, re.DOTALL)
    valid = []
    for obj in objects:
        try:
            item = json.loads(obj)
            if 'terme' in item:
                item.update({'doc_id': doc_id, 'modele': 'gemma-4-9b-it'})
                valid.append(item)
        except Exception:
            pass

    if valid:
        print(f"  [FALLBACK] {len(valid)} objets extraits")
        return valid

    print(f"  [ERROR] Parse échoué pour {doc_id}: {response[:200]}")
    return []

In [14]:
# ─── Pipeline LLaMA ──────────────────────────────────────────────────────────
all_definitions = []
results_by_doc = {}
timing_stats = []

for doc in tqdm(DOCUMENTS, desc="[Gemma-4] Extraction"):
    doc_path = Path(doc["path"])
    print(f"\n{'='*60}")
    print(f"[{doc['id']}] {doc['label']} ({doc['lang'].upper()})")

    if not doc_path.exists():
        print(f"  [SKIP]")
        continue

    text = extract_pdf_text(str(doc_path))
    if not text:
        continue

    t0 = time.time()
    defs = extract_definitions_llama(text, doc["lang"], doc["id"])
    total_time = time.time() - t0
    print(f"  → {len(defs)} définitions en {total_time:.1f}s")

    timing_stats.append({"doc_id": doc["id"], "time_s": total_time, "nb_defs": len(defs)})
    all_definitions.extend(defs)
    results_by_doc[doc["id"]] = {"label": doc["label"], "lang": doc["lang"],
                                  "nb_definitions": len(defs), "definitions": defs}

    with open(OUTPUT_DIR / f"{doc['id']}_definitions.json", "w", encoding="utf-8") as f:
        json.dump(defs, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "all_definitions_llama.json", "w", encoding="utf-8") as f:
    json.dump(all_definitions, f, ensure_ascii=False, indent=2)

if all_definitions:
    pd.DataFrame(all_definitions).to_csv(
        OUTPUT_DIR / "all_definitions_llama.csv", index=False, encoding="utf-8-sig")

print(f"\n✅ Total LLaMA : {len(all_definitions)} définitions")

[Gemma-4] Extraction:   0%|          | 0/9 [00:00<?, ?it/s]


[I001] Chalutage de fond (FR)
  Génération : 70.9s, 345 tokens
  → 3 définitions en 70.9s

[I002] Chasse à la baleine (EN)
  Génération : 349.8s, 2056 tokens
  → 24 définitions en 349.9s

[I003] Construction littorale (FR)
  Génération : 111.3s, 595 tokens
  → 6 définitions en 111.3s

[I004] Extraction de sable (FR)
  Génération : 257.6s, 1485 tokens
  → 17 définitions en 257.6s

[I005a] Rejet hydrocarbures - Protocole (FR)
  [SKIP]

[I005b] Rejet hydrocarbures - MARPOL Annexe I (FR)
  [SKIP]

[I006] Sacs plastique - MARPOL Annexe V (FR)
  Génération : 534.2s, 3138 tokens
  → 14 définitions en 534.3s

[I007] Peintures TBT - AFS Convention (EN)
  Génération : 166.6s, 935 tokens
  → 10 définitions en 166.6s

[I008] Oiseaux marins - CMS (FR)
  Génération : 228.3s, 1121 tokens
  → 11 définitions en 228.3s

✅ Total LLaMA : 85 définitions


In [17]:
import torch
import json
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import pdfplumber
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# --- Configuration ───────────────────────────────────────────────────────────
MODEL_NAME = "google/gemma-4-E4B-it"
# Note : nécessite un token HuggingFace avec accès Gemma-4
# from huggingface_hub import login; login(token="hf_VOTRE_TOKEN")

OUTPUT_DIR = Path("/content/drive/MyDrive/data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_DIR = Path("/content/drive/MyDrive/data/output/comparison")
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS = [
    {"id": "I005b", "label": "Rejet hydrocarbures - MARPOL Annexe I", "lang": "fr",
     "path": "/content/drive/MyDrive/data/raw/Rejet_dhydrocarbure/Annexe1consolidee_Marpol.pdf"},

]

GPU : Tesla T4


In [20]:
import json
from pathlib import Path

# Liste explicite des 9 fichiers à fusionner
target_files = [
    "I001_definitions.json", "I002_definitions.json", "I003_definitions.json",
    "I004_definitions.json", "I005a_definitions.json", "I005b_definitions.json",
    "I006_definitions.json", "I007_definitions.json", "I008_definitions.json"
]

input_dir = Path('/content')
output_file = input_dir / 'all_ministral.json'
all_ministral_defs = []

print(f"🚀 Début de la fusion de {len(target_files)} fichiers...")

for file_name in target_files:
    f_path = input_dir / file_name
    if f_path.exists():
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_ministral_defs.extend(data)
                print(f"✅ {file_name} ajouté ({len(data)} déf.)")
        except Exception as e:
            print(f"❌ Erreur lors de la lecture de {file_name}: {e}")
    else:
        print(f"⚠️ Fichier introuvable : {file_name}")

# Sauvegarde du résultat
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_ministral_defs, f, ensure_ascii=False, indent=2)

print(f"\n✨ Fusion terminée. Total : {len(all_ministral_defs)} définitions dans {output_file}")

🚀 Début de la fusion de 9 fichiers...
✅ I001_definitions.json ajouté (113 déf.)
✅ I002_definitions.json ajouté (2 déf.)
✅ I003_definitions.json ajouté (7 déf.)
✅ I004_definitions.json ajouté (21 déf.)
✅ I005a_definitions.json ajouté (2 déf.)
✅ I005b_definitions.json ajouté (89 déf.)
✅ I006_definitions.json ajouté (10 déf.)
✅ I007_definitions.json ajouté (23 déf.)
✅ I008_definitions.json ajouté (14 déf.)

✨ Fusion terminée. Total : 281 définitions dans /content/all_ministral.json


In [23]:
import json
from pathlib import Path

# Liste explicite des fichiers LlamaParser
target_files = [
    "/content/llama-extract-ext-4ku9p1z7rxzx2jysnumiknu0h4cw-ICRW_convention.json",
    "/content/llama-extract-ext-8q7ocj7nghc7mlep9tsgk6o21amb-AFS_convention.json",
    "/content/llama-extract-ext-8xf24yr75m36ysntu0dlg8ms5rau-CMS_oiseau_marin.json",
    "/content/llama-extract-ext-91p9w49lz996vklxw6v2tkjneddv-Annexe1consolidée Marpol.json",
    "/content/llama-extract-ext-iwr3cqqpyag5g0xz71omin9t21wu-ANNEXEV_marpol.json",
    "/content/llama-extract-ext-m90vyn97ft9jxuddnpons9ieth8l-clalut_fond.json",
    "/content/llama-extract-ext-yhh3eymmduv8w3xe8wra3wyu0zy4-convention_cbd.json",
    "/content/llama-extract-ext-zrcg2w835sualepd3gciczrdhe4j-protocole_GIZC.json"
]

output_file = Path('/content/all_llamaParser.json')
all_llama_defs = []

print(f"🚀 Début de la fusion LlamaParser ({len(target_files)} fichiers)...")

for f_path_str in target_files:
    f_path = Path(f_path_str)
    if f_path.exists():
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                content = json.load(f)

                defs_found = []
                if isinstance(content, list):
                    defs_found = content
                elif isinstance(content, dict):
                    # Stratégie 1: Racine directe
                    for key in ['data', 'definitions', 'glossary_entries', 'glossary_terms', 'results']:
                        if key in content and isinstance(content[key], list):
                            defs_found = content[key]
                            break

                    # Stratégie 2: Sous la clé 'data'
                    if not defs_found and 'data' in content and isinstance(content['data'], dict):
                         for key in ['definitions', 'glossary_entries', 'glossary_terms']:
                             if key in content['data'] and isinstance(content['data'][key], list):
                                 defs_found = content['data'][key]
                                 break

                if defs_found:
                    all_llama_defs.extend(defs_found)
                    print(f"✅ {f_path.name} : {len(defs_found)} déf. ajoutées")
                else:
                    print(f"⚠️ {f_path.name} : Aucune liste de définitions trouvée (vérifier la clé JSON)")
        except Exception as e:
            print(f"❌ Erreur sur {f_path.name}: {e}")
    else:
        print(f"⚠️ Fichier introuvable : {f_path_str}")

# Sauvegarde
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_llama_defs, f, ensure_ascii=False, indent=2)

print(f"\n✨ Fusion terminée. Total : {len(all_llama_defs)} définitions dans {output_file}")

🚀 Début de la fusion LlamaParser (8 fichiers)...
✅ llama-extract-ext-4ku9p1z7rxzx2jysnumiknu0h4cw-ICRW_convention.json : 24 déf. ajoutées
✅ llama-extract-ext-8q7ocj7nghc7mlep9tsgk6o21amb-AFS_convention.json : 10 déf. ajoutées
✅ llama-extract-ext-8xf24yr75m36ysntu0dlg8ms5rau-CMS_oiseau_marin.json : 12 déf. ajoutées
✅ llama-extract-ext-91p9w49lz996vklxw6v2tkjneddv-Annexe1consolidée Marpol.json : 63 déf. ajoutées
✅ llama-extract-ext-iwr3cqqpyag5g0xz71omin9t21wu-ANNEXEV_marpol.json : 15 déf. ajoutées
✅ llama-extract-ext-m90vyn97ft9jxuddnpons9ieth8l-clalut_fond.json : 103 déf. ajoutées
✅ llama-extract-ext-yhh3eymmduv8w3xe8wra3wyu0zy4-convention_cbd.json : 17 déf. ajoutées
✅ llama-extract-ext-zrcg2w835sualepd3gciczrdhe4j-protocole_GIZC.json : 6 déf. ajoutées

✨ Fusion terminée. Total : 250 définitions dans /content/all_llamaParser.json


In [18]:
# ─── Pipeline LLaMA ──────────────────────────────────────────────────────────
all_definitions = []
results_by_doc = {}
timing_stats = []

for doc in tqdm(DOCUMENTS, desc="[Gemma-4] Extraction"):
    doc_path = Path(doc["path"])
    print(f"\n{'='*60}")
    print(f"[{doc['id']}] {doc['label']} ({doc['lang'].upper()})")

    if not doc_path.exists():
        print(f"  [SKIP]")
        continue

    text = extract_pdf_text(str(doc_path))
    if not text:
        continue

    t0 = time.time()
    defs = extract_definitions_llama(text, doc["lang"], doc["id"])
    total_time = time.time() - t0
    print(f"  → {len(defs)} définitions en {total_time:.1f}s")

    timing_stats.append({"doc_id": doc["id"], "time_s": total_time, "nb_defs": len(defs)})
    all_definitions.extend(defs)
    results_by_doc[doc["id"]] = {"label": doc["label"], "lang": doc["lang"],
                                  "nb_definitions": len(defs), "definitions": defs}

    with open(OUTPUT_DIR / f"{doc['id']}_definitions.json", "w", encoding="utf-8") as f:
        json.dump(defs, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "all_definitions_llama.json", "w", encoding="utf-8") as f:
    json.dump(all_definitions, f, ensure_ascii=False, indent=2)

if all_definitions:
    pd.DataFrame(all_definitions).to_csv(
        OUTPUT_DIR / "all_definitions_llama.csv", index=False, encoding="utf-8-sig")

print(f"\n✅ Total gemma : {len(all_definitions)} définitions")

[Gemma-4] Extraction:   0%|          | 0/1 [00:00<?, ?it/s]


[I005b] Rejet hydrocarbures - MARPOL Annexe I (FR)
  Génération : 324.1s, 1694 tokens
  → 11 définitions en 324.2s

✅ Total gemma : 11 définitions


In [19]:
import json
from pathlib import Path

# Dossier contenant les résultats
OUTPUT_DIR = Path("/content/drive/MyDrive/data/output")

all_definitions = []

# Lister tous les fichiers de définitions individuels
def_files = list(OUTPUT_DIR.glob("*_definitions.json"))
print(f"🔍 {len(def_files)} fichiers trouvés.")

for f_path in def_files:
    # Éviter de lire le fichier de destination s'il existe déjà
    if f_path.name == "all_definitions_gema.json":
        continue

    try:
        with open(f_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                all_definitions.extend(data)
            print(f"✅ Chargé : {f_path.name} ({len(data)} déf.)")
    except Exception as e:
        print(f"❌ Erreur sur {f_path.name} : {e}")

# Sauvegarde du fichier consolidé
output_file = OUTPUT_DIR / "all_definitions_gemma.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_definitions, f, ensure_ascii=False, indent=2)

print(f"\n✨ Total : {len(all_definitions)} définitions sauvegardées dans {output_file}")

🔍 9 fichiers trouvés.
✅ Chargé : I001_definitions.json (3 déf.)
✅ Chargé : I002_definitions.json (24 déf.)
✅ Chargé : I003_definitions.json (6 déf.)
✅ Chargé : I004_definitions.json (17 déf.)
✅ Chargé : I006_definitions.json (14 déf.)
✅ Chargé : I007_definitions.json (10 déf.)
✅ Chargé : I008_definitions.json (11 déf.)
✅ Chargé : I005a_definitions.json (0 déf.)
✅ Chargé : I005b_definitions.json (11 déf.)

✨ Total : 96 définitions sauvegardées dans /content/drive/MyDrive/data/output/all_definitions_gemma.json
